# Build a small model with Gemini · Создаём небольшую модель с Gemini

**Working rescue notebook / Рабочий резервный блокнот.** First try the guided demonstration in a completely blank notebook. Use this copy if generation or debugging gets stuck. All code is visible so you can inspect it.

We extend the earlier reservoir-and-one-farm example to two farms with **small, invented numbers**, not a calibrated Zarafshan model or a reproduction of the earlier game's data. Every number below is a volume in **million cubic metres (Mm³)** for one monthly step, not m³/s.

Сначала попробуйте демонстрацию в пустом блокноте. Этот вариант поможет, если сгенерированный код не работает. Мы расширяем предыдущую схему «водохранилище и одно хозяйство» до двух хозяйств, используя **условные учебные числа**, а не данные Зарафшана. Объёмы — **млн м³ за месячный шаг**, не м³/с.

**Question / Вопрос:** When water is scarce, who receives less? / Кому достанется меньше воды при дефиците?

```text
river → reservoir → allocation → farm_a → downstream
                              ↘ farm_b → downstream
```

No evaporation, seepage, delays or return flow from consumed water. Farms consume up to their needs; surplus continues downstream. Reservoir overflow also travels through the allocation node. These are deliberate simplifications.

Нет испарения, фильтрации, задержки стока или возврата потреблённой воды. Хозяйства потребляют воду в пределах потребности; избыток идёт вниз по течению. Перелив также проходит через узел распределения. Это намеренные упрощения.

<p><img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/hydrosolutions_logo.png" alt="hydrosolutions" width="220"></p>
<p><b>hydrosolutions GmbH</b> · Venusstrasse 29, 8050 Zürich, Switzerland · <a href="https://www.hydrosolutions.ch">www.hydrosolutions.ch</a> · hs@hydrosolutions.ch · +41 43 535 05 80</p>

In [ ]:
# Run once in a fresh Colab runtime. Uses the same TaqSim version as this course.
%pip install -q numpy pandas matplotlib networkx ipython deap ctrl-freak
%pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
import taqsim

assert taqsim.__version__ == "0.1.4", "Restart the runtime after installing TaqSim 0.1.4."

## 1. State the intended system / Опишите систему

Six monthly steps. Start with 6 Mm³ stored; capacity 20; no inaccessible storage. Inflow: 8, 4, 2, 0, 10, 2. Farm A needs 4 each month, Farm B needs 3. Release up to their combined need of 7. Divide water in proportion to needs: 4/7 to A and 3/7 to B. Equal percentages of need are served when water is scarce.

Шесть месяцев. Начальный запас 6 млн м³; вместимость 20; мёртвый объём 0. Приток: 8, 4, 2, 0, 10, 2. Хозяйству A нужно 4 в месяц, B — 3. Выпускать не более общей потребности 7. Делить пропорционально потребностям: A — 4/7, B — 3/7. При дефиците удовлетворяется одинаковая доля потребности каждого хозяйства.

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from taqsim import Demand, Edge, Frequency, Sink, Source, Splitter, Storage, Strategy, TimeSeries, Timestep, WaterSystem
from taqsim.node import DeficitRecorded, NoLoss, WaterConsumed, WaterReceived, WaterReleased, WaterSpilled

INFLOW = [8.0, 4.0, 2.0, 0.0, 10.0, 2.0]
INITIAL_STORAGE = 6.0
CAPACITY = 20.0
NEED_A = 4.0
NEED_B = 3.0
SHARE_A = NEED_A / (NEED_A + NEED_B)

In [ ]:
@dataclass(frozen=True)
class ReleaseNeed(Strategy):
    __params__ = ()
    need: float

    def release(self, node: Storage, inflow: float, t: Timestep) -> float:
        # In TaqSim 0.1.4 this month's inflow is already stored at this point.
        return min(self.need, max(0.0, node.storage - node.dead_storage))


@dataclass(frozen=True)
class ShareWater(Strategy):
    __params__ = ("share_a",)
    __bounds__ = {"share_a": (0.0, 1.0)}
    share_a: float

    def split(self, node: Splitter, amount: float, t: Timestep) -> dict[str, float]:
        return {"farm_a": amount * self.share_a, "farm_b": amount * (1.0 - self.share_a)}


def build_model(share_a: float = SHARE_A) -> WaterSystem:
    """Build a fresh six-month teaching model for one allocation rule."""
    if not 0.0 <= share_a <= 1.0:
        raise ValueError("share_a must be between 0 and 1")
    model = WaterSystem(frequency=Frequency.MONTHLY)
    model.add_node(Source(id="river", inflow=TimeSeries(values=INFLOW)))
    model.add_node(
        Storage(
            id="reservoir",
            capacity=CAPACITY,
            initial_storage=INITIAL_STORAGE,
            dead_storage=0.0,
            release_policy=ReleaseNeed(NEED_A + NEED_B),
            loss_rule=NoLoss(),
        )
    )
    model.add_node(Splitter(id="allocation", split_policy=ShareWater(share_a)))
    model.add_node(
        Demand(
            id="farm_a", requirement=TimeSeries(values=[NEED_A] * len(INFLOW)), consumption_fraction=1.0, efficiency=1.0
        )
    )
    model.add_node(
        Demand(
            id="farm_b", requirement=TimeSeries(values=[NEED_B] * len(INFLOW)), consumption_fraction=1.0, efficiency=1.0
        )
    )
    model.add_node(Sink(id="downstream"))
    for source, target in [
        ("river", "reservoir"),
        ("reservoir", "allocation"),
        ("allocation", "farm_a"),
        ("allocation", "farm_b"),
        ("farm_a", "downstream"),
        ("farm_b", "downstream"),
    ]:
        model.add_edge(Edge(id=f"{source}_to_{target}", source=source, target=target))
    model.validate()
    return model

## 2. Run and account for every unit of water / Запустите и проверьте баланс воды

For each month: **starting storage + inflow = ending storage + consumption by both farms + downstream outflow**. A balanced model can still implement the wrong allocation rule.

Для каждого месяца: **начальный запас + приток = конечный запас + потребление двух хозяйств + сток вниз по течению**. Даже при правильном балансе правило распределения может быть неверным.

In [ ]:
def run_model(share_a: float = SHARE_A) -> pd.DataFrame:
    """Run from the same starting storage and check the monthly water balance."""
    model = build_model(share_a)
    model.simulate(len(INFLOW))

    def series(node_id: str, event: type, field: str = "amount") -> np.ndarray:
        trace = model.nodes[node_id].trace(event, field=field)
        return np.array([trace.get(t, 0.0) for t in range(len(INFLOW))])

    release = series("reservoir", WaterReleased)
    spill = series("reservoir", WaterSpilled)
    storage = INITIAL_STORAGE + np.cumsum(np.array(INFLOW) - release - spill)
    consumed_a = series("farm_a", WaterConsumed)
    consumed_b = series("farm_b", WaterConsumed)
    downstream = series("downstream", WaterReceived)
    start = np.r_[INITIAL_STORAGE, storage[:-1]]
    balance_error = start + INFLOW - storage - consumed_a - consumed_b - downstream
    result = pd.DataFrame(
        {
            "month": np.arange(1, len(INFLOW) + 1),
            "inflow": INFLOW,
            "storage_end": storage,
            "release": release,
            "spill": spill,
            "A_consumed": consumed_a,
            "B_consumed": consumed_b,
            "A_shortage": series("farm_a", DeficitRecorded, "deficit"),
            "B_shortage": series("farm_b", DeficitRecorded, "deficit"),
            "downstream": downstream,
            "balance_error": balance_error,
        }
    )
    assert np.allclose(balance_error, 0, atol=1e-9), "Water balance does not close."
    assert np.all((storage >= -1e-9) & (storage <= CAPACITY + 1e-9)), "Storage outside its limits."
    assert np.isclose(storage[-1], model.nodes["reservoir"].storage), "Storage record does not agree."
    return result


result = run_model()
display(result.round(3))

## 3. Check by hand / Проверьте вручную

Month 1: 6 + 8 = 14 available; release 7; store 7. A gets 4, B gets 3. Month 3: 4 + 2 = 6 available; A gets 24/7, B gets 18/7; storage reaches zero. Month 4 has no water.

Expected end storage: **7, 4, 0, 0, 3, 0**. Expected release: **7, 7, 6, 0, 7, 5**. Over all six months, initial storage 6 + inflow 26 = consumption 32. The total shortage is 10.

Месяц 1: доступно 6 + 8 = 14; выпуск 7; конечный запас 7. A получает 4, B — 3. Месяц 3: доступно 4 + 2 = 6; A получает 24/7, B — 18/7; запас падает до нуля. В месяце 4 воды нет.

Ожидаемый запас: **7, 4, 0, 0, 3, 0**. Выпуск: **7, 7, 6, 0, 7, 5**. За шесть месяцев: начальный запас 6 + приток 26 = потребление 32. Общий дефицит — 10.

These fixed checks apply to the original inputs only. When changing the system, calculate a new expected result; do not delete checks just to make the code pass.

Эти численные проверки относятся только к исходным данным. После изменения системы сначала рассчитайте новый ожидаемый результат.

In [ ]:
assert np.allclose(result.storage_end, [7, 4, 0, 0, 3, 0]), "Unexpected baseline storage."
assert np.allclose(result.release, [7, 7, 6, 0, 7, 5]), "Unexpected baseline release."
assert np.allclose(result.A_consumed / NEED_A, result.B_consumed / NEED_B), "Not equal fractions of need."
assert np.isclose(result.A_shortage.sum() + result.B_shortage.sum(), 10), "Unexpected total shortage."
assert np.isclose(result.A_consumed.sum() + result.B_consumed.sum(), 32), "Unexpected consumption."
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(result.month, result.storage_end, marker="o")
axes[0].set_ylabel("Storage / Запас, Mm³")
axes[1].bar(result.month - 0.18, result.A_shortage, width=0.36, label="Farm A / Хозяйство A")
axes[1].bar(result.month + 0.18, result.B_shortage, width=0.36, label="Farm B / Хозяйство B")
axes[1].set_ylabel("Shortage / Дефицит, Mm³")
axes[1].set_xlabel("Month / Месяц")
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. A deliberate modelling mistake / Намеренная ошибка

Ask Gemini to change the split to 50/50, calling it “equal treatment”. The code runs and the water balance closes. **Does it match equal fractions of each farm's need?** No: with 7 available, A receives 3.5 of its need of 4, while B needs only 3. The surplus 0.5 goes downstream. Correct the share to 4/7; do not alter the checks to accept the mistake.

Попросите Gemini разделить воду 50/50, назвав это «равным обеспечением». Код работает и баланс сходится. **Одинаковая ли доля потребности удовлетворяется?** Нет: при наличии 7 хозяйство A получает 3,5 при потребности 4, а B нужно лишь 3. Избыток 0,5 уходит вниз по течению. Исправьте долю A на 4/7; не меняйте проверки ради ошибочного результата.

In [ ]:
mistake = run_model(0.5)
assert not np.allclose(mistake.A_consumed / NEED_A, mistake.B_consumed / NEED_B), "Mistake was not detected."
corrected = run_model(SHARE_A)
assert np.allclose(corrected.A_consumed / NEED_A, corrected.B_consumed / NEED_B), "Correction did not work."
display(
    pd.DataFrame(
        {
            "case": ["50/50 mistake", "4/7 corrected"],
            "A_shortage": [mistake.A_shortage.sum(), corrected.A_shortage.sum()],
            "B_shortage": [mistake.B_shortage.sum(), corrected.B_shortage.sum()],
            "downstream": [mistake.downstream.sum(), corrected.downstream.sum()],
        }
    ).round(3)
)

## 5. Optional: make it your own / По желанию: измените модель

Choose **one** change: drier inflows; a different initial storage; priority for one farm; or a downstream minimum flow (which requires a new allocation rule). Predict the effect before asking Gemini to edit the code. Draw the new network if connections change. Repeat the water balance and a hand calculation. Explain who benefits, who loses, and what assumptions remain unrealistic. Do not treat these illustrative results as operational advice.

Выберите **одно** изменение: меньший приток; другой начальный запас; приоритет одному хозяйству; либо минимальный сток вниз по течению (нужно новое правило распределения). Сначала предскажите результат. Если меняются связи, нарисуйте новую схему. Повторите баланс и ручной расчёт. Объясните, кто выигрывает, кто проигрывает и какие допущения остаются нереалистичными. Учебные результаты не являются рекомендациями по эксплуатации.